# EEG Conformer — Cross-Subject

Cross-subject MEG brain-state classification with **ConformerCross**:
EEGConformerGAP with switchable spatial-conv normalisation (BatchNorm / InstanceNorm).

Pipeline:
1. Load cross-train and cross-test data via shared `protocol_utils`
2. LOSO validation over 2 training subjects to select `use_instance_norm`
3. Retrain on all cross-train files for the mean-best epoch
4. Evaluate on 3 unseen test subjects

## 1. Imports

In [ ]:
import sys, math
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader

sys.path.append('..')
from protocol_utils import (
    DatasetPaths, PreprocessConfig,
    list_files, make_windows, split_files_train_val,
    summarize_logits, CLASS_NAMES,
    file_accuracy_from_logits,
)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

from protocol_utils import make_leave_one_subject_out_folds

## 2. Cross Data Loading

Same preprocessing as intra: downsample ×4, z-score per sensor, windows 512 samples, stride 128.
Three separate test sets (one per unseen subject) are pre-windowed here.

In [ ]:
paths = DatasetPaths(data_root='../Final Project data')
cfg   = PreprocessConfig(downsample_factor=4, window_size=512, stride=128)

cross_train_files = list_files(paths.cross_train)
cross_test_files  = [
    ('test1', list_files(paths.cross_test1)),
    ('test2', list_files(paths.cross_test2)),
    ('test3', list_files(paths.cross_test3)),
]

cross_test_data = []
for name, files in cross_test_files:
    X_t, y_t, groups_t = make_windows(files, cfg)
    cross_test_data.append((name, X_t, y_t, groups_t))
    print(f'{name}: {X_t.shape}  ({len(files)} files)')

print(f'\nCross-train files: {len(cross_train_files)}')

## 3. LOSO Fold Construction

`make_leave_one_subject_out_folds` groups files by subject ID and returns 2 folds.
Each fold holds out one subject as validation and trains on the other.
`val_groups` tracks the source file of each window, enabling file-level accuracy.

In [ ]:
folds = make_leave_one_subject_out_folds(cross_train_files)

fold_data = []
for val_subject, fold_train_files, fold_val_files in folds:
    X_tr, y_tr, _         = make_windows(fold_train_files, cfg)
    X_vl, y_vl, vl_groups = make_windows(fold_val_files,   cfg)
    fold_data.append({
        'val_subject': val_subject,
        'X_train': X_tr, 'y_train': y_tr,
        'X_val':   X_vl, 'y_val':   y_vl, 'val_groups': vl_groups,
    })
    print(f'Fold val={val_subject}: train {X_tr.shape}, val {X_vl.shape}')

## 4. Model

ConformerCross adds a switchable normalisation to the spatial conv:
- **BatchNorm** (default): accumulates subject statistics across training — efficient but may not generalise
- **InstanceNorm**: normalises each sample independently — no accumulated subject statistics, more robust to unseen subjects

The temporal BatchNorm is left unchanged (z-score already normalises per-sensor amplitude).

In [ ]:
class ConvModule(nn.Module):
    def __init__(self, n_channels=248, n_filters=40, kernel_temp=25,
                 pool_size=75, pool_stride=15, dropout=0.5):
        super().__init__()
        self.temporal = nn.Sequential(
            nn.Conv2d(1, n_filters, kernel_size=(1, kernel_temp), bias=False),
            nn.BatchNorm2d(n_filters),
        )
        self.spatial = nn.Sequential(
            nn.Conv2d(n_filters, n_filters, kernel_size=(n_channels, 1), bias=False),
            nn.BatchNorm2d(n_filters),
            nn.ELU(),
            nn.AvgPool2d(kernel_size=(1, pool_size), stride=(1, pool_stride)),
            nn.Dropout(dropout),
        )
    def forward(self, x):
        x = self.temporal(x)
        x = self.spatial(x)
        x = x.squeeze(2)
        return x.permute(0, 2, 1)  # (B, T_seq, n_filters)


class ConformerCross(nn.Module):
    """EEGConformerGAP with switchable spatial-conv normalisation for cross-subject."""
    def __init__(self, n_channels=248, n_classes=4, window_size=512,
                 n_filters=40, kernel_temp=25, pool_size=75, pool_stride=15,
                 num_heads=8, num_layers=2, ff_dim=80, dropout=0.5,
                 use_instance_norm=False):
        super().__init__()
        self.temporal = nn.Sequential(
            nn.Conv2d(1, n_filters, kernel_size=(1, kernel_temp), bias=False),
            nn.BatchNorm2d(n_filters),
        )
        spatial_norm = (nn.InstanceNorm2d(n_filters, affine=True)
                        if use_instance_norm else nn.BatchNorm2d(n_filters))
        self.spatial = nn.Sequential(
            nn.Conv2d(n_filters, n_filters, kernel_size=(n_channels, 1), bias=False),
            spatial_norm, nn.ELU(),
            nn.AvgPool2d(kernel_size=(1, pool_size), stride=(1, pool_stride)),
            nn.Dropout(dropout),
        )
        T_prime = window_size - kernel_temp + 1
        T_seq   = (T_prime - pool_size) // pool_stride + 1
        self.pos_embedding = nn.Parameter(torch.randn(1, T_seq, n_filters) * 0.02)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=n_filters, nhead=num_heads,
            dim_feedforward=ff_dim, dropout=dropout,
            activation='gelu', batch_first=True,
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.norm        = nn.LayerNorm(n_filters)
        self.classifier  = nn.Linear(n_filters, n_classes)

    def forward(self, x):
        x = self.temporal(x)
        x = self.spatial(x)
        x = x.squeeze(2).permute(0, 2, 1)  # (B, T_seq, n_filters)
        x = x + self.pos_embedding
        x = self.transformer(x)
        x = self.norm(x)
        x = x.mean(dim=1)                   # GAP
        return self.classifier(x)


_m  = ConformerCross(kernel_temp=51).to(device)
_in = torch.zeros(2, 1, 248, 512).to(device)
print(f'ConformerCross parameters: {sum(p.numel() for p in _m.parameters()):,}')
print(f'Output shape: {tuple(_m(_in).shape)}')
del _m, _in

## 5. Training Utilities

In [ ]:
class MEGWindowDataset(Dataset):
    def __init__(self, X, y):
        self.X, self.y = X, y
    def __len__(self):
        return len(self.X)
    def __getitem__(self, idx):
        return (torch.from_numpy(self.X[idx]).unsqueeze(0),
                torch.tensor(self.y[idx], dtype=torch.long))


def run_epoch(loader, model, criterion, optimizer=None):
    training = optimizer is not None
    model.train() if training else model.eval()
    total_loss, correct, n = 0.0, 0, 0
    ctx = torch.enable_grad() if training else torch.no_grad()
    with ctx:
        for x, y in loader:
            x, y   = x.to(device), y.to(device)
            logits = model(x)
            loss   = criterion(logits, y)
            if training:
                optimizer.zero_grad(); loss.backward(); optimizer.step()
            total_loss += loss.item() * len(y)
            correct    += (logits.argmax(1) == y).sum().item()
            n          += len(y)
    return total_loss / n, correct / n


def get_val_file_acc(model, X_val, y_val, val_groups, batch_size=128):
    model.eval()
    all_logits = []
    with torch.no_grad():
        for i in range(0, len(X_val), batch_size):
            x = torch.from_numpy(X_val[i:i+batch_size]).unsqueeze(1).to(device)
            all_logits.append(model(x).cpu().numpy())
    return file_accuracy_from_logits(np.concatenate(all_logits, axis=0), y_val, val_groups)

## 6. Hyperparameters

In [ ]:
# Hyperparameters fixed from intra grid search; use_instance_norm selected by LOSO below
best_cross = {
    'kernel_temp': 51,
    'n_filters':   40,
    'dropout':     0.3,
    'use_instance_norm': False,  # updated by LOSO search below
    'mean_best_epoch':   13,     # updated by LOSO search below
}

X_cross_all, y_cross_all, _ = make_windows(cross_train_files, cfg)
cross_all_loader = DataLoader(MEGWindowDataset(X_cross_all, y_cross_all),
                              batch_size=64, shuffle=True, num_workers=0)
print(f'Full cross-train set: {X_cross_all.shape}  ({len(cross_train_files)} files)')

## 7. LOSO Grid Search — BatchNorm vs InstanceNorm

2 configs × 2 LOSO folds = 4 training runs.
Primary metric: mean **file-level accuracy** across folds.
`mean_best_epoch` = ceil of mean best epoch, used for the final retrain.

In [ ]:
def train_one_config_cross_conformer(use_instance_norm, fold_data,
                                     kernel_temp, n_filters, dropout, num_layers=2,
                                     max_epochs=30, patience=5,
                                     lr=5e-4, weight_decay=1e-4):
    fold_results = []
    for fold in fold_data:
        loader = DataLoader(MEGWindowDataset(fold['X_train'], fold['y_train']),
                            batch_size=64, shuffle=True, num_workers=0)
        model = ConformerCross(
            n_channels=248, n_classes=4, window_size=512,
            n_filters=n_filters, kernel_temp=kernel_temp,
            num_layers=num_layers, dropout=dropout,
            use_instance_norm=use_instance_norm,
        ).to(device)
        optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
        criterion = nn.CrossEntropyLoss()

        best_val_acc, best_epoch, patience_counter = 0.0, 0, 0
        for epoch in range(1, max_epochs + 1):
            run_epoch(loader, model, criterion, optimizer)
            val_file_acc = get_val_file_acc(model, fold['X_val'], fold['y_val'], fold['val_groups'])
            if val_file_acc > best_val_acc:
                best_val_acc, best_epoch, patience_counter = val_file_acc, epoch, 0
            else:
                patience_counter += 1
                if patience_counter >= patience:
                    break
        fold_results.append({'val_file_acc': best_val_acc, 'best_epoch': best_epoch})
        print(f'    fold val={fold["val_subject"]}: file_acc={best_val_acc:.3f}  epoch={best_epoch}')

    mean_val = float(np.mean([r['val_file_acc'] for r in fold_results]))
    mean_ep  = int(np.ceil(np.mean([r['best_epoch'] for r in fold_results])))
    return {'mean_val_file_acc': mean_val, 'mean_best_epoch': mean_ep}


print(f'Fixed: kernel_temp={best_cross["kernel_temp"]}, '
      f'n_filters={best_cross["n_filters"]}, dropout={best_cross["dropout"]}\n')

gs_results = []
for use_in in [False, True]:
    print(f'use_instance_norm={use_in}')
    result = train_one_config_cross_conformer(
        use_instance_norm=use_in,
        fold_data=fold_data,
        kernel_temp=best_cross['kernel_temp'],
        n_filters=best_cross['n_filters'],
        dropout=best_cross['dropout'],
    )
    gs_results.append({'use_instance_norm': use_in, **result})
    print(f'  -> mean_val_file_acc={result["mean_val_file_acc"]:.3f}  '
          f'mean_epoch={result["mean_best_epoch"]}\n')

gs_results.sort(key=lambda r: r['mean_val_file_acc'], reverse=True)
best_cross.update(gs_results[0])
print(f'Selected: use_instance_norm={best_cross["use_instance_norm"]}  '
      f'mean_val_file_acc={best_cross["mean_val_file_acc"]:.3f}  '
      f'mean_best_epoch={best_cross["mean_best_epoch"]}')

## 8. Final Training

Train on **all** cross-train files (both subjects, no val split) for `mean_best_epoch` epochs.
Hyperparameters and epoch count are fixed from the LOSO search above.

In [ ]:
final_cross_conformer = ConformerCross(
    n_channels=248, n_classes=4, window_size=512,
    n_filters=best_cross['n_filters'],
    kernel_temp=best_cross['kernel_temp'],
    dropout=best_cross['dropout'],
    num_layers=2,
    use_instance_norm=best_cross['use_instance_norm'],
).to(device)

optimizer = torch.optim.Adam(final_cross_conformer.parameters(), lr=5e-4, weight_decay=1e-4)
criterion = nn.CrossEntropyLoss()
n_epochs  = best_cross['mean_best_epoch']

print(f'Training ConformerCross for {n_epochs} epochs '
      f'(use_instance_norm={best_cross["use_instance_norm"]})...\n')
for epoch in range(1, n_epochs + 1):
    loss, win_acc = run_epoch(cross_all_loader, final_cross_conformer, criterion, optimizer)
    print(f'  Epoch {epoch:02d}/{n_epochs} | loss {loss:.4f} | window_acc {win_acc:.3f}')

torch.save(final_cross_conformer.state_dict(), 'conformer_cross_final.pt')
print('\nSaved conformer_cross_final.pt')

## 9. Test Evaluation — 3 Unseen Subjects

Each test subject is evaluated independently.
Primary metric: **mean file accuracy** across all 3 subjects.
Each test set has 16 files → accuracy moves in steps of 6.25%.

In [ ]:
def predict_logits_cross_conformer(windows):
    final_cross_conformer.eval()
    all_logits = []
    with torch.no_grad():
        for i in range(0, len(windows), 128):
            x = torch.from_numpy(windows[i:i+128]).unsqueeze(1).to(device)
            all_logits.append(final_cross_conformer(x).cpu().numpy())
    return np.concatenate(all_logits, axis=0)


print(f'{"Subject":<10} {"Window acc":>11} {"File acc":>10}')
print('-' * 34)
cross_conf_metrics = []
for name, X_t, y_t, groups_t in cross_test_data:
    m = summarize_logits(predict_logits_cross_conformer(X_t), y_t, groups_t)
    cross_conf_metrics.append((name, m))
    print(f'{name:<10} {m["window_acc"]:>11.3f} {m["file_acc"]:>10.3f}')

mean_conf = np.mean([m['file_acc'] for _, m in cross_conf_metrics])
print(f'\nMean file accuracy: {mean_conf:.3f}  <- primary metric')

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (name, m) in zip(axes, cross_conf_metrics):
    cm = m['file_confusion_matrix']
    im = ax.imshow(cm, cmap='Blues')
    ax.set_xticks(range(4)); ax.set_xticklabels(CLASS_NAMES, rotation=45, ha='right', fontsize=8)
    ax.set_yticks(range(4)); ax.set_yticklabels(CLASS_NAMES, fontsize=8)
    ax.set_xlabel('Predicted'); ax.set_ylabel('True')
    ax.set_title(f'ConformerCross {name}\nFile acc: {m["file_acc"]:.3f}')
    for i in range(4):
        for j in range(4):
            ax.text(j, i, cm[i, j], ha='center', va='center', fontsize=9,
                    color='white' if cm[i, j] > cm.max() / 2 else 'black')
    plt.colorbar(im, ax=ax)
plt.suptitle(f'Cross-Subject ConformerCross — Mean file accuracy: {mean_conf:.3f}',
             fontsize=12, y=1.02)
plt.tight_layout()
plt.show()